# Chapter 18 &mdash; Fixpoint Equations and the $Y$ Combinator

**Concept 7 of the Chapter 18 decomposition:** *Fixpoint Equations and the $Y$ Combinator*

$Y = \lambda f.(\lambda x.\,f(x\,x))(\lambda x.\,f(x\,x))$ satisfies $Y\,G = G\,(Y\,G)$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-Y-Combinator/Concept-Y-Combinator.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$$Y = \lambda f.\,(\lambda x.\,f\,(x\,x))\,(\lambda x.\,f\,(x\,x))$$

The defining property, by one $\beta$-step:

$$Y\,G \;\to\; (\lambda x.\,G\,(x\,x))(\lambda x.\,G\,(x\,x)) \;\to\; G\,\big((\lambda x.\,G\,(x\,x))(\lambda x.\,G\,(x\,x))\big) \;=\; G\,(Y\,G)$$

So $Y\,G$ **is** a fixpoint of $G$: exactly the $f$ with $f = G\,f$ that Concept 6
needed. Recursion, with **no names anywhere**.

The engine is **self-application** $x\,x$ &mdash; a term applying itself, which is what
lets a nameless function reach itself.

In a language with **eager** evaluation (Python, ML, Scheme), $Y$ as written diverges:
$x\,x$ is evaluated before it is needed. Concept 8 fixes that.

## 2. Definitions

### Y and its eager cousin

In [ ]:
# --- fixpoint combinators ------------------------------------------------
# Y diverges under Python's EAGER evaluation, because (x x) is evaluated
# before it is needed.  Y_e ("eager Y", also called Z) wraps the
# self-application in a lambda, delaying it until it is applied.
Y  = lambda f: (lambda x: f(x(x)))(lambda x: f(x(x)))          # loops in Python
Ye = lambda f: (lambda x: f(lambda v: x(x)(v)))(lambda x: f(lambda v: x(x)(v)))


G_fact = lambda f: lambda n: 1 if n == 0 else n * f(n - 1)

### Verifying the fixpoint property

In [ ]:
def is_fixpoint(G, f, inputs):
    return all(G(f)(n) == f(n) for n in inputs)

## 3. Tests

**$Y$ diverges in Python** &mdash; the very first thing to see.

In [ ]:
import sys
sys.setrecursionlimit(200)
try:
    Y(G_fact)
    print("no error (unexpected)")
except RecursionError:
    print("RecursionError -- Y(G) never returns under eager evaluation")
sys.setrecursionlimit(3000)

**$Y_e$** works: same idea, self-application delayed.

In [ ]:
fact = Ye(G_fact)
print([fact(n) for n in range(8)])
assert [fact(n) for n in range(8)] == [1, 1, 2, 6, 24, 120, 720, 5040]

And it really is a **fixpoint**: $G\,f$ and $f$ agree everywhere.

In [ ]:
print("is Ye(G) a fixpoint of G on 0..7 ?", is_fixpoint(G_fact, fact, range(8)))
assert is_fixpoint(G_fact, fact, range(8))
print("  G(fact)(5) =", G_fact(fact)(5), "   fact(5) =", fact(5))

**No names anywhere.** The whole thing is one expression.

In [ ]:
anonymous_factorial = (lambda f: (lambda x: f(lambda v: x(x)(v)))
                                 (lambda x: f(lambda v: x(x)(v))))(
                       lambda g: lambda n: 1 if n == 0 else n * g(n - 1))
print("5! =", anonymous_factorial(5))
print("8! =", anonymous_factorial(8))
assert anonymous_factorial(8) == 40320
print("\nNot one name is defined or referred to.  That is the point of Y.")

**Self-application** is the engine.

In [ ]:
selfapp = lambda x: x(x)
print("  (lambda x: x(x)) applied to the identity :", selfapp(lambda y: y)(7))
print()
print("x x is how a nameless term reaches itself.  It is also how you write")
print("Omega and diverge -- the same trick, used two ways.")

$Y$ works for **any** $G$, which is what makes it a combinator.

In [ ]:
G_fib  = lambda f: lambda n: n if n < 2 else f(n - 1) + f(n - 2)
G_sum  = lambda f: lambda n: 0 if n == 0 else n + f(n - 1)
G_len  = lambda f: lambda xs: 0 if not xs else 1 + f(xs[1:])
print("  fib  :", [Ye(G_fib)(n) for n in range(10)])
print("  sum  :", [Ye(G_sum)(n) for n in range(7)])
print("  len  :", Ye(G_len)([1, 2, 3, 4, 5]))
assert Ye(G_fib)(9) == 34 and Ye(G_sum)(5) == 15 and Ye(G_len)([1]*5) == 5

## 4. Exercises


1. Perform the $\beta$-reduction $Y\,G \to G\,(Y\,G)$ on paper.
2. Is $Y$ typeable in a simply-typed lambda calculus? Why not?
3. Find Turing's fixpoint combinator $\Theta$ and check it has the same property.

In [ ]:
# Your work for the exercises above.